# Simulating a qubit: predicting measurement statistics from a gate sequence

**Task.** A qubit starts in |0⟩, a sequence of gates is applied, and the qubit is measured. From the gate sequence
alone, predict the measurement statistics: the Bloch vector `r = (⟨X⟩, ⟨Y⟩, ⟨Z⟩)`, from which the Born rule gives
every outcome probability, `P(±) = (1 ± r_i)/2` in each basis. Models are trained on circuits of **1–40 gates**
and tested on circuits of up to **1000 gates**.

**Why this is the theory in action.** Single-qubit gates are unit quaternions (SU(2)), so a quaternion channel *is*
a qubit. The Bloch vector is quadratic in q, so the Born rule is exactly the sign-invariant readout `L(qqᵀ)`, and the
global phase −1 (physically unobservable) is the N of Theorem 1 that the readout must ignore.

**Three settings:**
* **Clifford** {H, S}: the single-qubit Clifford group, which acts on the Bloch sphere as the 24 rotations of a cube
  (S4, solvable, derived length 3). By Proposition 2, a stack of three commutative layers can represent it.
* **Clifford + T** {H, S, T}: dense in SU(2), so not solvable; no commutative stack can track it, one quaternion
  channel can.
* **Clifford + T, imperfect gates**: as on real hardware, every gate over-rotates slightly about a slightly tilted
  axis (a fixed, unknown error per gate). Labels come from the true gates; the models see only the gate names.
  A simulator that uses the ideal gates drifts; a model that learns the real gates should not. This is gate
  characterization learned from measurement data.

**Models** (same tokens, same training): HyperSSM (one quaternion channel, a transition per gate, readout linear in
qqᵀ: 4 numbers of state); a 3-layer selective complex diagonal stack (Mamba-style, the depth Proposition 2 needs for
the Clifford group); an LSTM; and the ideal-gate simulator (no learning). Every model uses the best of `N_INIT`
short starts by training loss.

In [ ]:
import math, time, json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

# ---------------- configuration ----------------
SETTINGS     = [("clifford", False), ("clifford_T", False), ("clifford_T", True)]   # (gate set, imperfect gates)
MODELS       = ["HyperSSM", "C-stack", "LSTM"]
SEEDS        = [0, 1, 2]
STEPS        = 3_000
BATCH        = 256
LR           = {"HyperSSM": 1e-2, "C-stack": 1e-3, "LSTM": 1e-3}
TRAIN_LEN    = (1, 40)
CURRICULUM   = 0.3            # fraction of training on circuits of 1–5 gates
VAL_LEN      = (40, 256)
TEST_LEN     = [1, 10, 40, 100, 256, 1000]
N_VAL, N_TEST = 512, 512
EVAL_EVERY   = 250
N_INIT       = 8              # multi-start, every model: best training loss after WARMUP_STEPS
WARMUP_STEPS = 200
OVER_ROTATION = 0.02          # imperfect gates: relative angle error (2%) ...
AXIS_TILT_DEG = 2.0           # ... and axis tilt (degrees), drawn once per gate
GATE_SEED    = 123

## Gates as unit quaternions (rotations of the Bloch sphere)

In [ ]:
def q_axis_angle(axis, angle):
    axis = np.asarray(axis, float); axis = axis / np.linalg.norm(axis)
    return np.concatenate([[np.cos(angle / 2)], np.sin(angle / 2) * axis])

def qmul_np(a, b):
    aw, ax, ay, az = np.moveaxis(a, -1, 0); bw, bx, by, bz = np.moveaxis(b, -1, 0)
    return np.stack([aw*bw - ax*bx - ay*by - az*bz, aw*bx + ax*bw + ay*bz - az*by,
                     aw*by - ax*bz + ay*bw + az*bx, aw*bz + ax*by - ay*bx + az*bw], -1)

IDEAL = {  # Bloch-sphere rotations (global phases are irrelevant)
    "H": q_axis_angle([1, 0, 1], math.pi),        # π about (x + z)/√2
    "S": q_axis_angle([0, 0, 1], math.pi / 2),    # π/2 about z
    "T": q_axis_angle([0, 0, 1], math.pi / 4),    # π/4 about z
}
GATESETS = {"clifford": ["H", "S"], "clifford_T": ["H", "S", "T"]}

def imperfect(gates, seed=GATE_SEED):
    """Each gate over-rotates by a fixed relative error and turns about a slightly tilted axis."""
    rng = np.random.default_rng(seed); out = {}
    for g in gates:
        q = IDEAL[g]; ang = 2 * math.acos(np.clip(q[0], -1, 1)); axis = q[1:] / np.linalg.norm(q[1:])
        tilt = np.cross(axis, rng.normal(size=3)); tilt /= np.linalg.norm(tilt)
        axis2 = np.cos(math.radians(AXIS_TILT_DEG)) * axis + np.sin(math.radians(AXIS_TILT_DEG)) * tilt
        out[g] = q_axis_angle(axis2, ang * (1 + OVER_ROTATION * rng.choice([-1, 1])))
    return out

def bloch(q):
    """Bloch vector of U|0⟩ for the rotation q (…, 4): R(q) applied to ẑ."""
    w, x, y, z = np.moveaxis(q, -1, 0)
    return np.stack([2 * (x * z + w * y), 2 * (y * z - w * x), 1 - 2 * (x * x + y * y)], -1)

def make_circuits(gates, true_q, lengths, n, rng):
    """Random circuits; label = Bloch vector after the true gates, applied in order (later gates on the left)."""
    if np.isscalar(lengths): L = np.full(n, lengths)
    else: L = rng.integers(lengths[0], lengths[1] + 1, n)
    T = int(L.max()); tok = rng.integers(0, len(gates), (n, T))
    tok[np.arange(T)[None] >= L[:, None]] = -1                       # padding
    Q = np.tile([1.0, 0, 0, 0], (n, 1)); G = np.stack([true_q[g] for g in gates])
    for t in range(T):
        live = tok[:, t] >= 0
        Q[live] = qmul_np(G[tok[live, t]], Q[live])
    return torch.from_numpy(tok), torch.from_numpy(L), torch.from_numpy(bloch(Q).astype(np.float32))

## Models

In [ ]:
def qmul(a, b):
    aw, ax, ay, az = a.unbind(-1); bw, bx, by, bz = b.unbind(-1)
    return torch.stack([aw*bw - ax*bx - ay*by - az*bz, aw*bx + ax*bw + ay*bz - az*by,
                        aw*by - ax*bz + ay*bw + az*bx, aw*bz + ax*by - ay*bx + az*bw], -1)
def qexp(v):
    th = v.norm(dim=-1, keepdim=True)
    return torch.cat([torch.cos(th), torch.sinc(th / math.pi) * v], -1)
def ordered_product(a):
    """a_T ⋯ a_1 (the HyperSSM recurrence q_t = a_t q_{t−1}), log-depth tree as in a parallel scan."""
    B, T, _ = a.shape; T2 = 1 << max(0, (T - 1).bit_length())
    if T2 > T:
        e = a.new_zeros(B, T2 - T, 4); e[..., 0] = 1; a = torch.cat([a, e], 1)
    while a.shape[1] > 1: a = qmul(a[:, 1::2], a[:, 0::2])
    return a[:, 0]

class HyperSSM(nn.Module):
    """One quaternion channel, tracking regime: q_t = exp(v_{g_t}) q_{t−1}, q_0 = 1; readout linear in qqᵀ."""
    def __init__(self, n_gates):
        super().__init__()
        self.v = nn.Parameter(torch.randn(n_gates, 3))
        iu = torch.triu_indices(4, 4); self.register_buffer("i0", iu[0]); self.register_buffer("i1", iu[1])
        self.head = nn.Linear(10, 3)
    def forward(self, tok, L):
        valid = (tok >= 0).float().unsqueeze(-1)
        q = ordered_product(qexp(self.v[tok.clamp(min=0)] * valid))
        return self.head((q.unsqueeze(-1) * q.unsqueeze(-2))[:, self.i0, self.i1])

def affine_scan(a, b):
    """h_t = a_t h_{t−1} + b_t, h_0 = 0, elementwise complex; Hillis–Steele scan, O(log T) depth."""
    T, d = a.shape[1], 1
    while d < T:
        a, b = (torch.cat([a[:, :d], a[:, d:] * a[:, :-d]], 1), torch.cat([b[:, :d], a[:, d:] * b[:, :-d] + b[:, d:]], 1))
        d *= 2
    return b

class CLayer(nn.Module):
    """Selective complex diagonal layer (Mamba-style): h_t = exp(Δ_t(−α + iθ_t)) h_{t−1} + Δ_t B x_t, plus MLP."""
    def __init__(self, d, K=32):
        super().__init__(); self.K = K
        self.norm1, self.norm2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.dt, self.theta, self.B = nn.Linear(d, K), nn.Linear(d, K), nn.Linear(d, 2 * K)
        self.log_alpha = nn.Parameter(torch.log(torch.rand(K) * 0.05 + 1e-3))
        self.out = nn.Linear(2 * K, d)
        self.mlp = nn.Sequential(nn.Linear(d, 2 * d), nn.GELU(), nn.Linear(2 * d, d))
    def forward(self, x, mask):
        u = self.norm1(x)
        dlt = F.softplus(self.dt(u)) * mask
        a = torch.exp(torch.complex(-dlt * self.log_alpha.exp(), dlt * self.theta(u)))
        b = torch.view_as_complex(self.B(u).view(*u.shape[:2], self.K, 2).contiguous()) * dlt
        h = affine_scan(a, b)
        x = x + self.out(torch.cat([h.real, h.imag], -1))
        return x + self.mlp(self.norm2(x))

class CStack(nn.Module):
    def __init__(self, n_gates, d=64, layers=3):
        super().__init__(); self.emb = nn.Embedding(n_gates + 1, d)
        self.layers = nn.ModuleList(CLayer(d) for _ in range(layers)); self.head = nn.Linear(d, 3)
    def forward(self, tok, L):
        mask = (tok >= 0).float().unsqueeze(-1)
        x = self.emb(torch.where(tok >= 0, tok + 1, 0))
        for layer in self.layers: x = layer(x, mask)
        return self.head(x[torch.arange(len(L)), L - 1])

class LSTMModel(nn.Module):
    def __init__(self, n_gates, hidden=128):
        super().__init__(); self.emb = nn.Embedding(n_gates + 1, 32)
        self.lstm = nn.LSTM(32, hidden, batch_first=True); self.head = nn.Linear(hidden, 3)
    def forward(self, tok, L):
        out, _ = self.lstm(self.emb(torch.where(tok >= 0, tok + 1, 0)))
        return self.head(out[torch.arange(len(L)), L - 1])

MAKE = {"HyperSSM": HyperSSM, "C-stack": CStack, "LSTM": LSTMModel}

def simulator(gates, q_used):
    """No learning: simulate with the given gates (the ideal ones, for the reference)."""
    G = torch.tensor(np.stack([q_used[g] for g in gates]), dtype=torch.float32)
    def f(tok, L):
        valid = (tok >= 0).float().unsqueeze(-1)
        a = G.to(tok.device)[tok.clamp(min=0)] * valid + (1 - valid) * torch.tensor([1.0, 0, 0, 0], device=tok.device)
        q = ordered_product(a); w, x, y, z = q.unbind(-1)
        return torch.stack([2 * (x * z + w * y), 2 * (y * z - w * x), 1 - 2 * (x * x + y * y)], -1)
    return f

## Training and evaluation
Loss: mean squared error on the Bloch vector. Error: the angle between predicted and true Bloch vectors (degrees);
infidelity `1 − F = (1 − cos angle)/2` for a pure state.

In [ ]:
def angle_deg(rp, rt):
    c = (F.normalize(rp, dim=-1) * rt).sum(-1).clamp(-1, 1)
    return torch.rad2deg(torch.arccos(c))

@torch.no_grad()
def errors(fn, data, bs=512):
    tok, L, R = data; out = []
    for i in range(0, len(L), bs):
        out.append(angle_deg(fn(tok[i:i+bs].to(DEVICE), L[i:i+bs].to(DEVICE)), R[i:i+bs].to(DEVICE)).cpu())
    return torch.cat(out).numpy()

def loss_fn(model, data):
    tok, L, R = data
    return F.mse_loss(model(tok.to(DEVICE), L.to(DEVICE)), R.to(DEVICE))

def train(name, gates, true_q, seed, val):
    rng = np.random.default_rng(1000 + seed)
    probe = make_circuits(gates, true_q, (1, 5), 2048, np.random.default_rng(5000 + seed))
    best_l, model = float("inf"), None
    for r in range(N_INIT):                                         # multi-start (same for every model)
        torch.manual_seed(1000 * seed + r); m = MAKE[name](len(gates)).to(DEVICE)
        opt = torch.optim.Adam(m.parameters(), lr=LR[name])
        for _ in range(WARMUP_STEPS):
            l = loss_fn(m, make_circuits(gates, true_q, (1, 5), BATCH, rng)); opt.zero_grad(); l.backward(); opt.step()
        with torch.no_grad(): l = loss_fn(m, probe).item()
        if l < best_l: best_l, model = l, m
    opt = torch.optim.AdamW(model.parameters(), lr=LR[name], weight_decay=0.0)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, STEPS)
    best, best_state, t0 = float("inf"), None, time.time()
    for step in range(1, STEPS + 1):
        hi = 5 if step < CURRICULUM * STEPS else TRAIN_LEN[1]
        loss = loss_fn(model, make_circuits(gates, true_q, (TRAIN_LEN[0], hi), BATCH, rng))
        opt.zero_grad(set_to_none=True); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); sched.step()
        if step % EVAL_EVERY == 0 and step >= CURRICULUM * STEPS:
            model.eval(); v = float(np.median(errors(model, val))); model.train()
            if v < best: best, best_state = v, {k: x.detach().clone() for k, x in model.state_dict().items()}
            if step % (4 * EVAL_EVERY) == 0:
                print(f"    {name:8s} seed {seed} step {step:5d} loss {loss.item():.5f} val median {v:7.3f}° "
                      f"(best {best:.3f}°) [{time.time()-t0:.0f}s]", flush=True)
    model.load_state_dict(best_state); model.eval()
    return model

In [ ]:
results, learned = {}, {}
for gs, noisy in SETTINGS:
    gates = GATESETS[gs]; true_q = imperfect(gates) if noisy else {g: IDEAL[g] for g in gates}
    key = gs + (" (imperfect gates)" if noisy else "")
    print(f"=== {key}: gates {gates}")
    val = make_circuits(gates, true_q, VAL_LEN, N_VAL, np.random.default_rng(7))
    tests = {n: make_circuits(gates, true_q, n, N_TEST, np.random.default_rng(100 + n)) for n in TEST_LEN}
    results[key] = {"ideal-gate simulator": {n: errors(simulator(gates, IDEAL), tests[n]) for n in TEST_LEN}}
    for name in MODELS:
        for seed in SEEDS:
            m = train(name, gates, true_q, seed, val)
            results[key][f"{name}|{seed}"] = {n: errors(m, tests[n]) for n in TEST_LEN}
            if name == "HyperSSM": learned[(key, seed)] = (m, gates, true_q)
            print(f"  {name:8s} seed {seed}: median error (°) " +
                  ", ".join(f"{n}: {np.median(results[key][f'{name}|{seed}'][n]):.2f}" for n in TEST_LEN), flush=True)

## Results: median error of the predicted Bloch vector against circuit length

In [ ]:
def summary(res, prefix):
    keys = [k for k in res if k == prefix or k.startswith(prefix + "|")]
    med = np.array([[np.median(res[k][n]) for n in TEST_LEN] for k in keys])
    return med.mean(0), med.std(0)

rows = ["ideal-gate simulator"] + MODELS
for key in results:
    print(f"\n{key}: median angle between predicted and true Bloch vectors (°), mean ± s.d. over seeds")
    print(f"{'model':22s}" + "".join(f"{n:>13d}" for n in TEST_LEN))
    for r in rows:
        mu, sd = summary(results[key], r)
        print(f"{r:22s}" + "".join(f"{m:8.2f}±{d:<4.2f}" if d > 0 else f"{m:13.2f}" for m, d in zip(mu, sd)))
json.dump({k: {r: [list(map(float, x)) for x in summary(v, r)] for r in rows} for k, v in results.items()} |
          {"lengths": TEST_LEN}, open("qubit_results.json", "w"))

from matplotlib import font_manager
for f in font_manager.findSystemFonts():
    if any(k in f.lower() for k in ("heros", "helvet", "liberationsans")):
        try: font_manager.fontManager.addfont(f)
        except Exception: pass
plt.rcParams.update({"font.family": ["TeX Gyre Heros", "Liberation Sans", "DejaVu Sans"], "font.size": 8.5,
                     "axes.linewidth": 0.8, "axes.edgecolor": "#555555", "pdf.fonttype": 42})
BLUE, FILL, DARK = "#2a5ca8", "#e8eef8", "#3a3a3a"
STY = {"ideal-gate simulator": dict(color=DARK, ls=(0, (4, 2)), lw=1.1),
       "C-stack": dict(color=DARK, lw=1.0, marker="s", ms=3.2, mfc="white", mew=0.9),
       "LSTM": dict(color=DARK, lw=1.0, marker="^", ms=3.6, mfc="white", mew=0.9),
       "HyperSSM": dict(color=BLUE, lw=2.0, marker="o", ms=3.4)}
LAB = {"HyperSSM": "HyperSSM, ℍ (4 numbers)", "C-stack": "ℂ diagonal, 3 layers", "LSTM": "LSTM (128 units)",
       "ideal-gate simulator": "simulator, ideal gates"}
TITLE = {"clifford": "Clifford {H, S}", "clifford_T": "Clifford + T",
         "clifford_T (imperfect gates)": "Clifford + T, imperfect gates"}
fig, axes = plt.subplots(1, len(results), figsize=(3.0 * len(results), 2.9), sharey=True)
axes = np.atleast_1d(axes)
for ax, key in zip(axes, results):
    ax.axvspan(TRAIN_LEN[0], TRAIN_LEN[1], color=FILL, lw=0, zorder=0)
    for r in ["ideal-gate simulator", "C-stack", "LSTM", "HyperSSM"]:
        mu, sd = summary(results[key], r)
        mu = np.maximum(mu, 1e-3)
        if sd.any(): ax.fill_between(TEST_LEN, np.maximum(mu - sd, 1e-3), mu + sd, color=STY[r]["color"], alpha=0.14, lw=0)
        ax.plot(TEST_LEN, mu, label=LAB[r], zorder=3 if r == "HyperSSM" else 2, **STY[r])
    ax.set_xscale("log"); ax.set_yscale("log"); ax.set_title(TITLE[key], fontsize=9)
    ax.set_xticks(TEST_LEN); ax.set_xticklabels([str(n) for n in TEST_LEN]); ax.set_xlabel("gates in the circuit")
    for s in ("top", "right"): ax.spines[s].set_visible(False)
axes[0].set_ylabel("median Bloch-vector error (°)")
h, l = axes[-1].get_legend_handles_labels()
fig.legend(h, l, loc="lower center", ncol=4, frameon=False, fontsize=7, bbox_to_anchor=(0.5, 0.0))
plt.tight_layout(rect=(0, 0.1, 1, 1)); plt.savefig("qubit.pdf"); plt.savefig("qubit.png", dpi=400); plt.show()

## Gate characterization: what HyperSSM learned
The readout can choose any orientation of the Bloch sphere, so the learned gates match the true ones up to a common
rotation. Frame-independent checks: each gate's rotation angle, and the angle between pairs of gate axes.

In [ ]:
def rot_angle(q): return math.degrees(2 * math.acos(min(1.0, abs(float(q[0])))))
def axis(q): v = np.asarray(q[1:], float); return v / np.linalg.norm(v) * (1 if q[0] >= 0 else -1)
def axis_angle(q1, q2): return math.degrees(math.acos(min(1.0, abs(float(np.dot(axis(q1), axis(q2)))))))
for (key, seed), (m, gates, true_q) in learned.items():
    if seed != SEEDS[0]: continue
    qs = qexp(m.v.detach().cpu()).numpy(); print(f"\n{key} (seed {seed})")
    for g, q in zip(gates, qs):
        print(f"  {g}: rotation learned {rot_angle(q):8.3f}°   true {rot_angle(true_q[g]):8.3f}°   "
              f"ideal {rot_angle(IDEAL[g]):8.3f}°")
    for i in range(len(gates)):
        for j in range(i + 1, len(gates)):
            print(f"  axes {gates[i]}–{gates[j]}: learned {axis_angle(qs[i], qs[j]):7.3f}°   "
                  f"true {axis_angle(true_q[gates[i]], true_q[gates[j]]):7.3f}°")